# 02 - Model Comparison

Comparative evaluation of ML models for product category prediction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier


## Load and clean dataset

In [ ]:
df = pd.read_csv('../data/products.csv')
df.columns = df.columns.str.strip()

df = df.dropna(subset=['Product Title','Category Label'])

category_counts = df['Category Label'].value_counts()
rare = category_counts[category_counts < 2].index
df = df[~df['Category Label'].isin(rare)]

print(df.shape)
df.head()


## Train/Test Split

In [ ]:
X = df['Product Title'].astype(str)
y = df['Category Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Evaluate Multiple Models

In [ ]:
results = []
predictions = {}

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)

    results.append([name, acc])
    predictions[name] = pred

    print(f'{name}: {acc:.4f}')


In [ ]:
nb_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])
evaluate_model('MultinomialNB', nb_model)

lr_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000))
])
evaluate_model('LogisticRegression', lr_model)

svc_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC())
])
evaluate_model('LinearSVC', svc_model)

lgbm_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LGBMClassifier(verbose=-1))
])
evaluate_model('LightGBM', lgbm_model)


## Compare Accuracy

In [ ]:
results_df = pd.DataFrame(results, columns=['Model','Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)
display(results_df)

sns.barplot(data=results_df, x='Model', y='Accuracy')
plt.xticks(rotation=20)
plt.show()


## Classification Report and Confusion Matrix

In [ ]:
best_model = results_df.iloc[0]['Model']
best_pred = predictions[best_model]

print('Best model:', best_model)
print(classification_report(y_test, best_pred))

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots(figsize=(10,10))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
plt.show()


## Conclusion

Models compared:
- MultinomialNB
- Logistic Regression
- LinearSVC
- LightGBM

The final model is selected based on highest accuracy and classification metrics.
